In [ ]:
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs

In [ ]:
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],  # Falls back to DejaVu if Times isn't installed
        "axes.labelsize": 10,
        "font.size": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.figsize": (5.5, 3.5),
        "figure.dpi": 300,
    }
)

In [ ]:
DATA_DIR = Path("../../data")
MEDS_DIR = DATA_DIR / "MEDS_cohort" / "data"

## Raw MEDS

In [ ]:
demo_size = pl.read_csv(DATA_DIR / "raw_input" / "demo_subject_id.csv")
full_size = 299_712
multiplier = full_size / len(demo_size)
multiplier

In [ ]:
meds_data = pl.read_parquet(MEDS_DIR / "train" / "0.parquet")

In [ ]:
print(f"Total number of events in the demo dataset: {len(meds_data)}")
print(f"Estimated total events after scaling: {len(meds_data) * round(multiplier)}")

In [ ]:
meds_data = meds_data.with_columns(pl.col("code").str.split("//", inclusive=False).list.get(0).alias("prefix"))
prefix_counts = meds_data["prefix"].value_counts().sort("count", descending=False)
prefix_counts = prefix_counts.with_columns((pl.col("count") / len(meds_data) * 100).alias("percentage"))

meds_data

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))

bars = ax.barh(prefix_counts["prefix"], prefix_counts["percentage"], color="#4C72B0", edgecolor="none", height=0.7)
# ax.set_xscale('log')

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)  # Remove y-axis spine
ax.spines["bottom"].set_linewidth(0.5)
ax.tick_params(axis="y", length=0)

ax.xaxis.grid(True, linestyle="--", alpha=0.5, color="gray")
ax.set_axisbelow(True)

ax.set_xlabel("Percentage")

plt.tight_layout()

In [ ]:
events_per_subject_id = meds_data.group_by("subject_id").agg(pl.count("code").alias("num_events"))
events_per_subject_id.describe()

In [ ]:
fig, ax = plt.subplots()
ax.hist(events_per_subject_id["num_events"], bins=50, color="#4C72B0", edgecolor="white", linewidth=0.5)

ax.set_yscale("log")

ax.set_xlabel("Number of Events per Subject")
ax.set_ylabel("Frequency")

ax.grid(axis="y", linestyle="--", alpha=0.5, color="gray")
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.tight_layout()
plt.show()

## Tasks

In [ ]:
task_dfs = []
for file in (DATA_DIR / "aces_labels").glob("*/*/*.parquet"):
    print(file)
    df = pl.read_parquet(file)
    df = df.with_columns(pl.lit(file.parent.parent.name).alias("task"), pl.lit(file.parent.name).alias("split"))
    task_dfs.append(df)
task_df = pl.concat(task_dfs, how="vertical")
task_df

In [ ]:
task_count_df = (
    task_df.select("task", "split", "subject_id")
    .pivot(on="split", index="task", aggregate_function="len")
    .select("task", "train", "tuning", "held_out")
    .sort("task")
)
task_count_df

In [ ]:
task_full_df = task_count_df.with_columns(cs.numeric().mul(multiplier).round(0).cast(pl.Int64))
task_full_df

In [ ]:
positive_task_percentages = (
    task_df.filter(pl.col("split") == "train")
    .group_by("task")
    .agg((pl.col("boolean_value").mean() * 100).round(2).alias("positive_percentage"))
    .sort("task")
)
positive_task_percentages

## Structured Data

In [ ]:
structured_tokenized_dataset = datasets.load_from_disk(
    DATA_DIR / "structured" / "pretraining" / "tokenized_dataset" / "train"
)

In [ ]:
num_tokens = sum(len(example["input_ids"]) for example in structured_tokenized_dataset)
print(f"Total number of tokens in the demo dataset: {num_tokens}")
print(f"Estimated total tokens after scaling: {num_tokens * round(multiplier)}")

In [ ]:
from google.genai.local_tokenizer import LocalTokenizer

prompt_lengths = []
tokenizer = LocalTokenizer("gemini-3-pro-preview")
for file in (DATA_DIR / "natural_language" / "gemini" / "tuning").iterdir():
    print(file)
    gemini_dataset = datasets.load_from_disk(file)
    prompt_lengths.extend([tokenizer.count_tokens(prompt).total_tokens for prompt in gemini_dataset["prompt"]])

total_prompt_lengths = sum(prompt_lengths)

In [ ]:
MAX_OUTPUT_TOKENS = 1024
GEMINI_PRICE_INPUT_TOKENS_PER_1M = 2
GEMINI_PRICE_OUTPUT_TOKENS_PER_1M = 12

estimated_cost = (total_prompt_lengths / 1_000_000) * GEMINI_PRICE_INPUT_TOKENS_PER_1M + len(prompt_lengths) * (
    MAX_OUTPUT_TOKENS / 1_000_000
) * GEMINI_PRICE_OUTPUT_TOKENS_PER_1M

print(f"Total number of tokens in the Gemini prompts: {total_prompt_lengths}")
print(f"Estimated cost for the Gemini prompts: ${estimated_cost:.2f}")

print(f"Estimated total tokens in the Gemini prompts after scaling: {total_prompt_lengths * round(multiplier)}")
print(f"Estimated cost for the Gemini prompts after scaling: ${estimated_cost * round(multiplier):.2f}")

In [ ]:
# prompts_dfs = []
# for file in structured_prompt_directory.glob(f"*/{split}/*.parquet"):
#     print(file)
#     df = pl.read_parquet(file)
#     task = file.parent.parent.name
#     df = df.with_columns(pl.lit(task).alias("task"), pl.lit("Structured").alias("Method"))
#     prompts_dfs.append(df)

In [ ]:
# structured_prompt_directory = DATA_DIR / "structured" / "prompts"
# split = "tuning"

# prompts_dfs = []
# for file in structured_prompt_directory.glob(f"*/{split}/*.parquet"):
#     print(file)
#     df = pl.read_parquet(file)
#     task = file.parent.parent.name
#     df = df.with_columns(pl.lit(task).alias("task"), pl.lit("Structured").alias("Method"))
#     prompts_dfs.append(df)

In [ ]:
# nl_directory = Path("../../outputs/inference/natural_language/thinking/20260504_174759/tuning")

# for file in nl_directory.glob("*/*.parquet"):
#     print(file)
#     df = pl.read_parquet(file)
#     task = file.parent.name
#     df = df.with_columns(pl.lit(task).alias("task"), pl.lit("Natural Language").alias("Method"))
#     prompts_dfs.append(df)

In [ ]:
# prompts_df = pl.concat(prompts_dfs, how="diagonal")
# prompts_df

In [ ]:
# structured_prompts_df = structured_prompts_df.with_columns(
#     pl.col("ehr_text")
#     .str.split("\n")
#     .list.eval(pl.element().filter(pl.element().str.starts_with("#").not_()))
#     .list.len()
#     .alias("num_events")
# )
# structured_prompts_df

In [ ]:
# from datetime import timedelta

# from mimic_struct_predict.models.structured.inference_generative_rollout import parse_delta_line, parse_time_line


# def calculate_input_window(row: dict) -> timedelta:
#     ehr, pred = row["ehr_text"], row["prediction_time"]

#     time, day_delta = (0, 0), 0
#     time_found = False

#     # .splitlines() is cleaner than .split('\n')
#     for line in ehr.splitlines():
#         if not time_found and line.startswith("### Time//"):
#             time = parse_time_line(line, time_prefix="### Time//", time_unspecified="Unspecified", delimiter="//")
#             time_found = True
#         elif line.startswith("## Delta//"):
#             day_delta += parse_delta_line(line, delta_prefix="## Delta//")

#     # Calculate days earlier, then replace hours and minutes in one step
#     earliest_code_datetime = (pred - timedelta(days=day_delta)).replace(
#         hour=time[0], minute=time[1], second=0, microsecond=0
#     )

#     return pred - earliest_code_datetime


# structured_prompts_df = structured_prompts_df.with_columns(
#     input_window=pl.struct(["ehr_text", "prediction_time"]).map_elements(
#         calculate_input_window, return_dtype=pl.Duration
#     )
# )
# structured_prompts_df

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 6))

# # Using the same dataframe 'df' from the previous example
# sns.boxplot(data=structured_prompts_df, x="task", y="num_events")

# plt.ylabel("Number of Events in Timeline")
# plt.xlabel("Task")

# plt.tight_layout()
# plt.show()